In [19]:
import numpy as np
import pandas as pd
import os
import glob

# Define the required time range
start_time = 55563.250000
end_time = 56138.250000
num_lines = 576  # Fixed number of rows
expected_time = np.linspace(start_time, end_time, num_lines)

def fill_gaps(existing_time, data_array, expected_time):
    filled_data = np.full_like(expected_time, np.nan, dtype=np.float64)

    # Map existing values
    existing_indices = np.searchsorted(expected_time, existing_time)
    filled_data[existing_indices] = data_array

    # Fill missing values using the mean of 3 above and 3 below
    for i in range(len(filled_data)):
        if np.isnan(filled_data[i]):
            lower_bound = max(0, i - 3)
            upper_bound = min(len(filled_data), i + 4)
            valid_values = filled_data[lower_bound:upper_bound]
            valid_values = valid_values[~np.isnan(valid_values)]
            if len(valid_values) > 0:
                filled_data[i] = np.mean(valid_values)

    return filled_data

# Define the root directory and output directory
root_directory = r"../output/Tohoku_foreshock/Removal/RAW_data/PCA"
output_directory = r"../output/Tohoku_foreshock/Removal/Combined/PCA"
os.makedirs(output_directory, exist_ok=True)

# Group files by station name (E, N, U components have the same station name)
station_files = {}

# Iterate over the root directory and search for relevant files in After_ica subdirectories
for component_dir in ["E_MOM", "N_MOM", "U_MOM"]:
    # Search for all .mom files in subdirectories of E_MOM, N_MOM, U_MOM
    search_pattern = os.path.join(root_directory, "**", component_dir, "After_pca*", "*.mom")
    
    for file_path in glob.glob(search_pattern, recursive=True):
        base_name = os.path.basename(file_path).replace(".mom", "")
        
        if base_name not in station_files:
            station_files[base_name] = {"E": None, "N": None, "U": None}

        if component_dir == "E_MOM":
            station_files[base_name]["E"] = file_path
        elif component_dir == "N_MOM":
            station_files[base_name]["N"] = file_path
        elif component_dir == "U_MOM":
            station_files[base_name]["U"] = file_path

# Debugging: Print files found
print("\nFiles organized by station:")
for station_name, components in station_files.items():
    print(f"Station: {station_name}")
    for component, file_path in components.items():
        print(f"  {component}: {file_path}")

# Process each station's files
for station_name, files in station_files.items():
    print(f"\nProcessing station: {station_name}")

    # Read data from each component file (E, N, U)
    data = {}

    for component, file_path in files.items():
        if file_path is None:
            print(f"  Warning: Missing {component} file for station {station_name}")
            continue
        print(f"  Reading {component} file: {file_path}")
        try:
            df = pd.read_csv(file_path, sep='\\s+', header=None, names=["Time", component])
            print(f"    Data preview for {component}:")
            print(df.head())  # Print preview of data
            df = df.sort_values(by="Time")  # Ensure time is sorted
            data[component] = df
        except Exception as e:
            print(f"    Error reading {file_path}: {e}")

    # Ensure all components are aligned with the expected time values
    df_filled = pd.DataFrame({"Time": expected_time})

    for component in ["E", "N", "U"]:
        if component in data:
            print(f"  Filling gaps for {component}")
            values = data[component]["Time"].values
            component_values = data[component][component].values
            df_filled[component] = fill_gaps(values, component_values, expected_time)

    # Save the output file in the output directory
    output_file_path = os.path.join(output_directory, f"{station_name}_combined.dat")
    try:
        np.savetxt(output_file_path, df_filled.values, fmt="%.6f", delimiter=" ")
        print(f"  Saved: {output_file_path}")
    except Exception as e:
        print(f"  Error saving file: {e}")


Files organized by station:
Station: J134
  E: ../output/Tohoku_foreshock/Removal/RAW_data/PCA/E_MOM/After_pca71/J134.mom
  N: ../output/Tohoku_foreshock/Removal/RAW_data/PCA/N_MOM/After_pca81/J134.mom
  U: ../output/Tohoku_foreshock/Removal/RAW_data/PCA/U_MOM/After_pca88/J134.mom
Station: G174
  E: ../output/Tohoku_foreshock/Removal/RAW_data/PCA/E_MOM/After_pca71/G174.mom
  N: ../output/Tohoku_foreshock/Removal/RAW_data/PCA/N_MOM/After_pca81/G174.mom
  U: ../output/Tohoku_foreshock/Removal/RAW_data/PCA/U_MOM/After_pca88/G174.mom
Station: I011
  E: ../output/Tohoku_foreshock/Removal/RAW_data/PCA/E_MOM/After_pca71/I011.mom
  N: ../output/Tohoku_foreshock/Removal/RAW_data/PCA/N_MOM/After_pca81/I011.mom
  U: ../output/Tohoku_foreshock/Removal/RAW_data/PCA/U_MOM/After_pca88/I011.mom
Station: I005
  E: ../output/Tohoku_foreshock/Removal/RAW_data/PCA/E_MOM/After_pca71/I005.mom
  N: ../output/Tohoku_foreshock/Removal/RAW_data/PCA/N_MOM/After_pca81/I005.mom
  U: ../output/Tohoku_foreshock/Rem